Here you will find the connection  codes to sqlite and creation of the db employee_system.db then creation of tables !!

In [15]:
import sqlite3  # Import SQLite module

# Connect to database (creates file if not exists)
conn = sqlite3.connect("employee_system.db")

# Create a cursor object to execute SQL commands
cursor = conn.cursor()



In [2]:
def create_tables():
    conn = sqlite3.connect("employee_system.db")
    cursor = conn.cursor()

    # Create Employees Table
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS employees (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            payroll_number TEXT UNIQUE NOT NULL,  -- Supports mixed letters & numbers (e.g., IMC234)
            first_name TEXT NOT NULL,
            last_name TEXT NOT NULL,
            work_email TEXT UNIQUE NOT NULL,
            contact_info TEXT CHECK(contact_info LIKE '+254%') NOT NULL,  -- Must start with +254
            department TEXT CHECK(department IN ('IT', 'COLGATE', 'Finance', 'ADMIN', 'BTC')) NOT NULL,  -- Specific departments
            role TEXT CHECK(role IN ('Employee', 'HR')) NOT NULL,  -- Restricts to Employee or HR
            password TEXT NOT NULL,  -- Will store hashed passwords
            leave_entitlement INTEGER DEFAULT 36,  -- Total leave days per year
            leave_balance INTEGER DEFAULT 36  -- Remaining leave days
        )
    ''')

    conn.commit()

In [3]:
cursor.execute("PRAGMA table_info(employees);")
print(f' Columns in employee table are \n:',cursor.fetchall())  # This will show all columns in the employees table

 Columns in employee table are 
: [(0, 'id', 'INTEGER', 0, None, 1), (1, 'payroll_number', 'TEXT', 1, None, 0), (2, 'first_name', 'TEXT', 1, None, 0), (3, 'last_name', 'TEXT', 1, None, 0), (4, 'work_email', 'TEXT', 1, None, 0), (5, 'contact_info', 'TEXT', 0, None, 0), (6, 'department', 'TEXT', 1, None, 0), (7, 'role', 'TEXT', 1, None, 0), (8, 'password', 'TEXT', 1, None, 0), (9, 'leave_entitlement', 'INTEGER', 0, '36', 0), (10, 'leave_balance', 'INTEGER', 0, '36', 0)]


In [4]:
def create_attendance_table ():
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS attendance (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            employee_id INTEGER NOT NULL,
            timestamp DATETIME DEFAULT CURRENT_TIMESTAMP,
            event_type TEXT CHECK(event_type IN ('clock_in', 'clock_out')) NOT NULL,
            FOREIGN KEY (employee_id) REFERENCES employees(id) ON DELETE CASCADE
        )
    ''')
    
    conn.commit()
print('Attendance table created successfully!')

create_attendance_table()




Attendance table created successfully!


In [14]:
cursor.execute("PRAGMA table_info(attendance);")
print(f' Columns in attendance table are \n:',cursor.fetchall()) 

 Columns in attendance table are 
: [(0, 'id', 'INTEGER', 0, None, 1), (1, 'employee_id', 'INTEGER', 1, None, 0), (2, 'timestamp', 'DATETIME', 0, 'CURRENT_TIMESTAMP', 0), (3, 'event_type', 'TEXT', 1, None, 0)]


In [9]:
def create_leave_request_table():
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS leave_requests (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            employee_id INTEGER NOT NULL,
            leave_type TEXT CHECK(leave_type IN ('Annual', 'Sick', 'Unpaid', 'Maternity', 'Paternity')) NOT NULL,
            start_date DATE NOT NULL,
            end_date DATE NOT NULL,
            status TEXT CHECK(status IN ('pending', 'approved', 'rejected')) DEFAULT 'pending',
            remarks TEXT,
            reason TEXT,
            FOREIGN KEY (employee_id) REFERENCES employees(id) ON DELETE CASCADE
        )
    ''')
    
    conn.commit()
    print('Leave request table created successfully!')

create_leave_request_table()


Leave request table created successfully!


In [14]:
cursor.execute("PRAGMA table_info(leave_requests);")
print(f' Columns in leave request table are \n:',cursor.fetchall()) 

 Columns in leave request table are 
: [(0, 'id', 'INTEGER', 0, None, 1), (1, 'employee_id', 'INTEGER', 1, None, 0), (2, 'leave_type', 'TEXT', 1, None, 0), (3, 'start_date', 'DATE', 1, None, 0), (4, 'end_date', 'DATE', 1, None, 0), (5, 'status', 'TEXT', 0, "'pending'", 0), (6, 'remarks', 'TEXT', 0, None, 0), (7, 'reason', 'TEXT', 0, None, 0)]


In [13]:
import sqlite3

# Connect to the database
conn = sqlite3.connect("employee_system.db")
cursor = conn.cursor()

# Function to add the reason column to the leave_requests table
def add_reason_column():
    try:
        cursor.execute("ALTER TABLE leave_requests ADD COLUMN reason TEXT")
        conn.commit()
        print('Reason column added successfully!')
    except sqlite3.OperationalError as e:
        print(f"Error: {e}")

add_reason_column()



Error: duplicate column name: reason


In [5]:
import pandas as pd  # Import pandas for better table display

# Query to fetch all employee data
query = "SELECT * FROM employees"


# Load data into a pandas DataFrame
df = pd.read_sql_query(query, conn)

# Display the DataFrame
print(f'Employee data in the employees table is \n :{df}')

Employee data in the employees table is 
 :   id payroll_number first_name last_name               work_email  \
0   3        IMC1234      Peter     Adams  peter.adams@example.com   
1   4          XY120    Paulina   Nyokabi  eve.nyoksds@example.com   
2   5           5524        Ann       Kai   annkai.ak.14@gmail.com   
3   6            123       Girl    Friend          girl@friend.com   
4   7           5678        Ann    Mwaura            ann@gmail.com   
5   8           1234        Ann       Kai           test@gmail.com   

     contact_info    department      role     password  leave_entitlement  \
0      0791234567  Supply Chain  Employee  testp123ass                 36   
1    +25472290544         Legal        HR    testingss                 36   
2      0701001372            IT  Employee      pass123                 36   
3   0878962541313         Legal        HR     PASSY123                 36   
4   +254701001372            HR        HR    Pass@1234                 36   
5  +

In [5]:
import sqlite3

def authenticate_user(email, password):
    conn = sqlite3.connect("employee_system.db")
    cursor = conn.cursor()
    
    cursor.execute("SELECT * FROM employees WHERE work_email = ? AND password = ?", (email, password))
    user = cursor.fetchone()
    
    if user:
        print(f"Login successful for {user[1]} {user[2]}")  # Assuming index 1 is first name, index 2 is last name
        return user
    else:
        print("Invalid email or password")
        return None

# Example test case
authenticate_user('eve.adams@example.com', 'testpass')


Login successful for XYZ789 John


(1,
 'XYZ789',
 'John',
 'Peter',
 'eve.adams@example.com',
 '0791234567',
 'Sales',
 'Employee',
 'testpass',
 36,
 36)